In [1]:
import cobra
import sys
sys.path.append('../')
from modelfunctions import *
import os

# TODO remove and clean up
from memote.support.consistency import check_stoichiometric_consistency
import memote.support.consistency_helpers as con_helpers
from models_emil import * 

wd = os.path.abspath(os.getcwd()).removesuffix('Code/modelUpdates')
figdir = wd + 'Figures/'
models_dir = f'{wd}Data/pcm/'

In [388]:
def met_overview(pcm, rid):
    return pd.DataFrame({m.id:{'Formula': m.formula, 'factor': f, 'Charge': m.charge, 'Name': m.name, 'nRxns': len(m.reactions)} for m,f in get_rid(pcm, rid).metabolites.items()}).T.sort_values('factor')

def check_num_imbalanced(pcm, balance_with_protons = False, return_imbalanced = False):
    c = 0
    #slime_reactions = []
    imbalanced = []
    for r in pcm.reactions:
        # skip import export exchane biomass balance
        if any(r.id.startswith(prefix) for prefix in ['Im_', 'Ex_', 'Exch_', 'DM_', 'Sk_']) or \
                r.id in ['PigmentPool', 'BiomassRxn', 'precursorPool', 'ProteinPool', 'nucleotidePool']:
            continue

        # These do not really count since they only supply the biomass reaction
        if r.id.startswith('SLIMEr'):
            #slime_reactions.append(r)
            continue

        try:
            imba = r.check_mass_balance()
        except ValueError as e:
            c += 1
            if return_imbalanced:
                imbalanced.append(r)
            continue

        # 'X' refers to photons, and they cannot be balanced
        # ignore charge imbalances for now
        if not set(imba.keys()).difference(['X']):
            continue

        if imba and not (balance_with_protons and set(imba.keys()) == {'H', 'charge'} and imba['H'] == imba['charge']):
            if return_imbalanced:
                imbalanced.append(r)
            c += 1
    if return_imbalanced:
        return c, imbalanced
    return c

In [3]:
pcm1 = cobra.io.read_sbml_model(models_dir + 'pcm.v1.xml')

Set parameter Username
Set parameter LicenseID to value 2852561
Academic license - for non-commercial use only - expires 2027-08-10


In [383]:
pcm = cobra.io.read_sbml_model(models_dir + 'pcm.v2.xml')

In [389]:
check_num_imbalanced(pcm, return_imbalanced=True) # TODO

(0, [])

In [ ]:
# TODO
# check if R01277 is still in there; it shouldnt
# deal with dead-end metabolites
# unblock all reactions
# check stoichiometric consistency

In [390]:
deadM = {m for m in pcm.metabolites if len(m.reactions) == 1}
rs = {r for m in deadM for r in m.reactions}

In [391]:
len(rs)

115

In [392]:
print_rxns(sorted(rs, key=lambda r:r.id, reverse=True))

R11227
O2, oxygen + 2.0 H+, proton + 2.0 Reduced ferredoxin + Pheophorbide a --> H2O, water + 2.0 Oxidized ferredoxin + Epoxypheophorbide a
C00007[h] + 2.0 C00080[h] + 2.0 C00138[h] + C18021[h] --> C00001[h] + 2.0 C00139[h] + C21192[h]

R10710
6-Geranylgeranyl-2-methylbenzene-1,4-diol + S-Adenosylmethionine <=> H+, proton + 6-Geranylgeranyl-2,3-dimethylbenzene-1,4-diol + Adenosylhomocysteine
C20737[h] + aMet[h] <=> C00080[h] + C20738[h] + aH-Cys[h]

R10563
Nicotinamide adenine dinucleotide phosphate - reduced + H+, proton + L-Glyceraldehyde --> Nicotinamide adenine dinucleotide phosphate + Glycerol
C00005[h] + C00080[h] + C02426[h] --> C00006[h] + C00116[h]

R10054
Lysine + Nicotinamide adenine dinucleotide --> Nicotinamide adenine dinucleotide - reduced + H+, proton + (S)-2,3,4,5-Tetrahydropyridine-2-carboxylate + Ammonia
Lys[h] + NAD[h] --> C00004[h] + C00080[h] + C00450[h] + NH4[h]

R10049
Fructose 1,6-bisphosphate + Methylglyoxal --> 2.0 H+, proton + Glyceraldehyde 3-phosphate + 6-

In [399]:
met_overview(pcm, 'R01277')

,Formula,factor,Charge,Name,nRxns
NAD[h],C21H26N7O14P2,-1.0,-1,Nicotinamide adenine dinucleotide,97
CoA[h],C21H32N7O16P3S,-1.0,-4,Coenzyme A,137
C00517[h],C16H32O,-1.0,0,Hexadecanal,2
C00004[h],C21H27N7O14P2,1.0,-2,Nicotinamide adenine dinucleotide - reduced,89
C00080[h],H,1.0,1,"H+, proton",702
C00154[h],C37H62N7O17P3S,1.0,-4,Palmitoyl-CoA,2


In [412]:
get_rid(pcm, 'R01468')

Reaction identifier,R01468
Name,ATP:ethanolamine O-phosphotransferase
Memory address,0x7a384bd7bda0
Stoichiometry,"C00002[h] + C00080[h] + C00189[h] --> C00008[h] + C00346[h] Adenosine 5-triphosphate + H+, proton + Ethanolamine --> Adenosine 5-diphosphate + Ethanolamine phosphate"
GPR,Potri.002G063700 or Potri.005G197500 or Potri.003G193000 or Potri.001G032000 or Potri.006G120700
Lower bound,0.0
Upper bound,1000.0


In [417]:
print_rxns_mid(pcm, 'C00154[h]')

R01277: C00517[h] + CoA[h] + NAD[h] --> C00004[h] + C00080[h] + C00154[h]
Hexadecanal + Coenzyme A + Nicotinamide adenine dinucleotide --> Nicotinamide adenine dinucleotide - reduced + H+, proton + Palmitoyl-CoA

R01274: C00001[h] + C00154[h] --> C00080[h] + C16_0[h] + CoA[h]
H2O, water + Palmitoyl-CoA --> H+, proton + [FA0101] FA 16:0 + Coenzyme A



In [348]:
check_production(pcm, 'C00459[h]')

252.42499999999998

In [351]:
get_rid(pcm, 'BiomassRxn').metabolites

{<Metabolite C00001[h] at 0x7a3860e866c0>: -5.0,
 <Metabolite C00002[h] at 0x7a385a760980>: -10.0,
 <Metabolite NAD[h] at 0x7a3860e84b30>: -1.0,
 <Metabolite C00005[h] at 0x7a3860e866f0>: -6.0,
 <Metabolite C00007[h] at 0x7a385a7610a0>: -2.5,
 <Metabolite C00031[h] at 0x7a384f383650>: -1.0,
 <Metabolite C00080[h] at 0x7a384f380d10>: -5.0,
 <Metabolite pPg[h] at 0x7a384e27f440>: -1.0,
 <Metabolite pPr[h] at 0x7a384e27fdd0>: -1.0,
 <Metabolite Lipid[h] at 0x7a384e2efe00>: -1.0,
 <Metabolite starch5[h] at 0x7a384e19bef0>: -1.0,
 <Metabolite pPc[h] at 0x7a384e1d0350>: -1.0,
 <Metabolite C00006[h] at 0x7a385a762450>: 5.0,
 <Metabolite C00008[h] at 0x7a3860e84620>: 10.0,
 <Metabolite C00009[h] at 0x7a385a763080>: 10.0}

In [355]:
get_mid(pcm, 'pPc[h]')

Metabolite identifier,pPc[h]
Name,Secondary metabolites and precursors prod. in...
Memory address,0x7a384e1d0350
Formula,None
Compartment,h
In 2 reaction(s),"precursorPool, BiomassRxn"


In [358]:
find_group(pcm, get_rid(pcm, 'precursorPool'))[0].name

'pseudo reaction'

In [381]:
match_mname(pcm, 'dATP')

C00131[h] dATP


[<Metabolite C00131[h] at 0x7a384c6ac680>]

In [297]:
print_rxns_mid(pcm, 'C00365[h]')

R06613: C00005[h] + C00080[h] + C00143[h] + C00365[h] --> C00006[h] + C00101[h] + C00364[h]
Nicotinamide adenine dinucleotide phosphate - reduced + H+, proton + 5,10-Methylenetetrahydrofolate + dUMP --> Nicotinamide adenine dinucleotide phosphate + 5,6,7,8-Tetrahydrofolate + dTMP

R02100: C00001[h] + C00080[h] + C00460[h] --> C00013[h] + C00365[h]
H2O, water + H+, proton + dUTP --> Diphosphate, Pyrophosphate + dUMP

R02099: C00002[h] + C00080[h] + C00526[h] --> C00008[h] + C00365[h]
Adenosine 5-triphosphate + H+, proton + Deoxyuridine --> Adenosine 5-diphosphate + dUMP

R01663: C00001[h] + C00080[h] + C00239[h] --> C00365[h] + NH4[h]
H2O, water + H+, proton + dCMP --> dUMP + Ammonia



In [296]:
print_rxns_mid(pcm, 'C00705[h]')

R02024: C00001[h] + C00343[h] + C00705[h] --> 3.0 C00080[h] + C00112[h] + C00342[h]
H2O, water + Thioredoxin disulfide + dCDP --> 3.0 H+, proton + CDP + Thioredoxin

R02326: C00002[h] + C00080[h] + C00705[h] <=> C00008[h] + C00458[h]
Adenosine 5-triphosphate + H+, proton + dCDP <=> Adenosine 5-diphosphate + dCTP



In [197]:
get_mid(pcm, 'C00475[h]')

Metabolite identifier,C00475[h]
Name,Cytidine
Memory address,0x7a385b725520
Formula,C9H13N3O5
Compartment,h
In 4 reaction(s),"R00517, R01878, R00513, R00511"


In [162]:
get_mid(pcm, 'CAs[h]')

Metabolite identifier,CAs[h]
Name,N-Carbamoyl-L-aspartate
Memory address,0x7a385b67eff0
Formula,C5H6N2O5
Compartment,h
In 3 reaction(s),"R01993, Tr_Cas_h, R01397"


In [195]:
get_mid(pcm, 'PRPP[c]')

Metabolite identifier,PRPP[c]
Name,5-Phosphoribosyl 1-pyrophosphate
Memory address,0x7a385ac9ff50
Formula,C5H8O14P3
Compartment,c
In 2 reaction(s),"Tr_PRPP, Ex_PRPP"


In [193]:
print_rxns_mid(pcm, 'Ura[c]')

Tr_Ura: C00080[c] + Ura[c] --> C00080[h] + Ura[h]
H+, proton + Uracil --> H+, proton + Uracil

Im_Ura:  --> Ura[c]
 --> Uracil



In [192]:
print_rxns_mid(pcm, 'PRPP[c]')

Tr_PRPP: PRPP[h] --> PRPP[c]
5-Phosphoribosyl 1-pyrophosphate --> 5-Phosphoribosyl 1-pyrophosphate

Ex_PRPP: PRPP[c] --> 
5-Phosphoribosyl 1-pyrophosphate --> 



In [ ]:
with pcm:
    for r in pcm.reactions:
        if r.id.startswith('Im_'):
            pass#r.knock_out()
        elif r.id.startswith('Exch_'):
            pass#r.knock_out()
    res = check_production(pcm, 'H-Cys[h]')#, add_import=['C00073[h]'])
res